# Session 26: GPUs

A GPU has thousands of simple cores and its own memory. Today we use one from Python with CuPy, which runs the same
calls as NumPy on the GPU, and we measure when it pays off and what can go wrong.

- **Part 1**: your first GPU job (in class).
- **Part 2**: timing GPU code.
- **Part 3**: small and large arrays.
- **Part 4**: one kernel instead of six.
- **Part 5**: float32 or float64?
- **Part 6**: an example from a project (Exercise 3).

Run the cells in order, from top to bottom: click a cell and press Shift+Enter, on Mac and on Windows. When the text
says **Predict**, decide on your answer before you run the next cell.

## Before you start

Look at the top right corner of this notebook. It must say **OSPC (Python 3.12)**. If it says something else, click
on that name and choose OSPC (Python 3.12) from the list.

Your JupyterLab session has no GPU. Every GPU part runs as a small batch job that asks for one GPU:

```
sbatch -p gpu --gres=gpu:1 -c 4 -t 5
```

The whole class shares 2 GPUs at a time, so your job may wait in the queue for a few minutes. While it waits, you
can see it with `squeue --me` (state `PD`, pending) in a terminal. The next cell puts the course Python in the
`PATH` for the jobs.

In [ ]:
import os

os.environ["PATH"] = "/data/software/envs/ospc/bin:" + os.environ["PATH"]
print("CPUs of your session:", len(os.sched_getaffinity(0)))

## Part 1. Your first GPU job

The next cell writes `gpu_lab.py`, the script that every part runs with a different mode. Read the function
`first()`: it makes two arrays of 100 million numbers with NumPy, copies them to the GPU with `cp.asarray`, adds them
there, and copies the result back with `cp.asnumpy`. The code that adds them is the same as in NumPy: `a + b`.

The functions `gpu_ms` and `cpu_ms` time a call. Why `gpu_ms` needs `benchmark()` is the subject of Part 2.

In [ ]:
%%writefile gpu_lab.py
"""Small GPU experiments for Session 26.

    python gpu_lab.py MODE

MODE is one of: first, timing, size, fuse, precision, similarity. Run it in a batch job with one GPU, e.g.
    sbatch -p gpu --gres=gpu:1 -c 4 -t 5 --wait -o first.out --wrap "python gpu_lab.py first"
"""
import os
import platform
import sys
import time

import numpy as np
import cupy as cp
from cupyx.profiler import benchmark


def gpu_ms(fn, *args, repeat=20):
    """Median time of fn(*args) on the GPU, in ms. benchmark() warms up first and uses the GPU's own clock."""
    r = benchmark(fn, args, n_repeat=repeat, n_warmup=3)
    return float(np.median(r.gpu_times[0])) * 1e3


def cpu_ms(fn, *args, repeat=3):
    """Best time of fn(*args) on the CPU, in ms."""
    best = float("inf")
    for _ in range(repeat):
        t = time.perf_counter()
        fn(*args)
        best = min(best, time.perf_counter() - t)
    return best * 1e3


def header():
    props = cp.cuda.runtime.getDeviceProperties(0)
    free, total = cp.cuda.runtime.memGetInfo()
    print(f"node {platform.node()}, {len(os.sched_getaffinity(0))} CPUs, GPU: {props['name'].decode()}, "
          f"{props['multiProcessorCount']} SMs, {total / 1e9:.0f} GB of memory", flush=True)


def first():
    """CuPy runs the same calls as NumPy, on the GPU."""
    n = 100_000_000
    rng = np.random.default_rng(0)
    a = rng.random(n, dtype=np.float32)
    b = rng.random(n, dtype=np.float32)

    copies = []
    for _ in range(2):
        t = time.perf_counter()
        a_gpu = cp.asarray(a)              # copy a to the GPU's memory
        b_gpu = cp.asarray(b)
        cp.cuda.Device().synchronize()
        copies.append((time.perf_counter() - t) * 1e3)

    c_gpu = a_gpu + b_gpu                  # runs on the GPU
    c = cp.asnumpy(c_gpu)                  # copy the result back
    print("CuPy gives the same result as NumPy:", np.array_equal(c, a + b))

    t_cpu = cpu_ms(np.add, a, b)
    t_gpu = gpu_ms(cp.add, a_gpu, b_gpu)
    print(f"a + b on {n:,} numbers: NumPy {t_cpu:.1f} ms, CuPy {t_gpu:.2f} ms, "
          f"{t_cpu / t_gpu:.0f} times faster")
    print(f"copying a and b to the GPU: {copies[0]:.0f} ms the first time, {copies[1]:.0f} ms the second")


def timing():
    """The clock must wait for the GPU."""
    dev = cp.cuda.Device()
    a = cp.random.random((4000, 4000))     # float64
    dev.synchronize()

    t = time.perf_counter()
    c = a @ a
    dev.synchronize()
    print(f"first a @ a:                      {(time.perf_counter() - t) * 1e3:8.2f} ms")

    t = time.perf_counter()
    c = a @ a
    wrong = time.perf_counter() - t        # the clock stops before the GPU has finished
    dev.synchronize()
    right = time.perf_counter() - t
    print(f"a @ a, clock read at once:        {wrong * 1e3:8.3f} ms")
    print(f"a @ a, clock read after the sync: {right * 1e3:8.2f} ms")
    print(f"a @ a, benchmark():               {gpu_ms(cp.matmul, a, a, repeat=5):8.2f} ms")

    x = cp.arange(10_000_000, dtype=cp.float32)
    dev.synchronize()
    for i in range(3):
        t = time.perf_counter()
        y = cp.hypot(x, 2 * x)
        dev.synchronize()
        print(f"cp.hypot, call {i + 1}:                 {(time.perf_counter() - t) * 1e3:8.2f} ms")


def size():
    """Where does the GPU start to win? The data are already on the GPU."""
    print(f"{'numbers':>13} {'NumPy ms':>10} {'GPU ms':>9} {'GPU faster':>11}")
    for k in range(3, 9):
        n = 10**k
        a = cp.random.random(n, dtype=cp.float32)
        b = cp.random.random(n, dtype=cp.float32)
        a_cpu, b_cpu = cp.asnumpy(a), cp.asnumpy(b)
        t_cpu = cpu_ms(np.add, a_cpu, b_cpu, repeat=200 if n < 10**6 else 5)
        t_gpu = gpu_ms(cp.add, a, b, repeat=50)
        print(f"{n:>13,} {t_cpu:10.4f} {t_gpu:9.4f} {t_cpu / t_gpu:10.1f}x")


def length(x, y, z):
    return cp.sqrt(x * x + y * y + z * z)


def fuse():
    """Six kernels against one."""
    n = 100_000_000
    x, y, z = (cp.random.random(n, dtype=cp.float32) for _ in range(3))
    length_fused = cp.fuse(kernel_name="length_fused")(length)
    print("same result:", bool(cp.allclose(length(x, y, z), length_fused(x, y, z))))
    t1 = gpu_ms(length, x, y, z)
    t2 = gpu_ms(length_fused, x, y, z)
    xc, yc, zc = cp.asnumpy(x), cp.asnumpy(y), cp.asnumpy(z)
    t3 = cpu_ms(lambda: np.sqrt(xc * xc + yc * yc + zc * zc))
    gb1, gb2 = 14 * 4 * n / 1e9, 4 * 4 * n / 1e9
    print(f"6 kernels: {t1:7.2f} ms, {gb1:.1f} GB read and written, {gb1 / t1:.2f} GB per ms")
    print(f"fused:     {t2:7.2f} ms, {gb2:.1f} GB read and written, {gb2 / t2:.2f} GB per ms")
    print(f"NumPy:     {t3:7.1f} ms")


def precision():
    """float32 is fast on these GPUs; how wrong is it?"""
    a64 = cp.random.random((4000, 4000))
    a32 = a64.astype(cp.float32)
    for a in (a32, a64):
        t = gpu_ms(cp.matmul, a, a, repeat=5)
        print(f"a @ a, {a.dtype}: {t:7.2f} ms, {2 * 4000**3 / t / 1e9:6.1f} TFLOP/s")
    c64, c32 = a64 @ a64, a32 @ a32
    err = float(cp.abs(c32 - c64).max() / cp.abs(c64).max())
    print(f"largest difference float32 against float64, relative: {err:.1e}")
    x = cp.random.random(100_000_000, dtype=cp.float32)
    exact = float(x.astype(cp.float64).sum())
    print(f"sum of 1e8 float32 numbers on the GPU: {float(x.sum()):,.1f} (float64: {exact:,.1f})")


def similarity(x):
    """Cosine similarity between the columns (items) of x. Works with NumPy and CuPy arrays."""
    xp = cp.get_array_module(x)           # numpy or cupy, whichever x belongs to
    norms = xp.sqrt((x * x).sum(axis=0))
    norms[norms == 0] = 1
    xn = x / norms
    return xn.T @ xn


def ratings(users=50_000, items=5_000):
    """About 5% of the ratings are known, from 1 to 5 stars; the rest are 0."""
    rng = np.random.default_rng(7)
    known = rng.random((users, items), dtype=np.float32) < 0.05
    x = np.zeros((users, items), dtype=np.float32)
    x[known] = rng.integers(1, 6, size=int(known.sum()))
    return x


def similarity_run():
    x = ratings()
    print(f"ratings: {x.shape[0]:,} users x {x.shape[1]:,} items, {x.nbytes / 1e9:.1f} GB")
    similarity(x[:1000])                   # warm up
    t = time.perf_counter()
    s_cpu = similarity(x)
    t_cpu = time.perf_counter() - t
    print(f"CPU, {len(os.sched_getaffinity(0))} CPUs: {t_cpu * 1e3:.0f} ms")

    dev = cp.cuda.Device()
    similarity(cp.asarray(x[:1000]))      # warm up the kernels
    cp.asarray(x)                          # warm up the copy: the first big copy of a program is slow
    cp.get_default_memory_pool().free_all_blocks()
    dev.synchronize()
    t0 = time.perf_counter()
    x_gpu = cp.asarray(x)
    dev.synchronize()
    t1 = time.perf_counter()
    s_gpu = similarity(x_gpu)
    dev.synchronize()
    t2 = time.perf_counter()
    s = cp.asnumpy(s_gpu)
    t3 = time.perf_counter()
    print(f"GPU: copy in {(t1 - t0) * 1e3:.0f} ms, compute {(t2 - t1) * 1e3:.0f} ms, "
          f"copy out {(t3 - t2) * 1e3:.0f} ms, in all {(t3 - t0) * 1e3:.0f} ms")
    print(f"GPU {t_cpu / (t3 - t0):.0f} times faster; largest difference: {np.abs(s - s_cpu).max():.1e}")


MODES = {"first": first, "timing": timing, "size": size, "fuse": fuse, "precision": precision,
         "similarity": similarity_run}

if __name__ == "__main__":
    header()
    MODES[sys.argv[1] if len(sys.argv) > 1 else "first"]()

**Predict**: how many times faster than NumPy will the GPU add the two arrays, once they are in its memory?
And how long will it take to copy them there (800 MB)? The script copies them twice.

The job takes about 10 seconds once it starts. The first line of the output says which node and which GPU ran it.

In [ ]:
!sbatch -p gpu --gres=gpu:1 -c 4 -t 5 -o first.out --wait --wrap "python gpu_lab.py first"
!cat first.out

On julia (RTX PRO 6000 Blackwell) the addition took 0.81 ms against 36 ms with NumPy, 45 times faster; on haskell
(RTX 6000 Ada) 1.45 ms, about 34 times faster. Copying the two arrays in took about as long as NumPy's whole
addition, even the second time (40 ms on julia, 56 ms on haskell). Add the copy of the result back, and a program
that only adds two arrays is faster on the CPU (Lecture 13). The first copy
of a program is slower still, by about 0.2 s, while the GPU's driver sets itself up. Warm up the copies too before
you time them.

## Part 2. Timing GPU code

CuPy returns from a call as soon as the GPU has been told what to do, long before the GPU has finished. A clock read
right after the call measures almost nothing. `cp.cuda.Device().synchronize()` waits until the GPU is done.

Read the function `timing()`. **Predict**: what will the clock say for `a @ a` (4000 x 4000, float64) when it is read
at once, and after the synchronization? Which of the three calls of `cp.hypot` will be the slowest?

In [ ]:
!sbatch -p gpu --gres=gpu:1 -c 4 -t 5 -o timing.out --wait --wrap "python gpu_lab.py timing"
!cat timing.out

On julia: 0.36 ms at once, 84 ms after the synchronization. The first `a @ a` is slower because CuPy starts the
matrix library (cuBLAS) the first time it is used. The first call of `cp.hypot` is slow too: CuPy writes a small
program for the GPU (a **kernel**) and compiles it, about 0.1 s per kernel. It keeps the compiled kernel in `~/.cupy`,
so if you run the cell again, the first call only loads it from disk and takes about a millisecond.

The rules: warm up first, synchronize before you read the clock, and repeat. `cupyx.profiler.benchmark` does all
three, and reads the GPU's own clock.

## Part 3. Small and large arrays

Every kernel costs the CPU about 5 microseconds to launch, whatever the size of the array. NumPy adds 1,000 numbers
in half a microsecond.

**Predict**: from what size does the GPU win? The data are already on the GPU, so there are no copies.

In [ ]:
!sbatch -p gpu --gres=gpu:1 -c 4 -t 5 -o size.out --wait --wrap "python gpu_lab.py size"
!cat size.out

On julia the GPU took about 6.5 microseconds for 1,000 numbers and for 10,000: that is the launch, not the work. It
wins from about 100,000 numbers (by 1.4 times), and by 45 times at 100 million. A GPU needs big arrays: a few thousand numbers do not give its
cores enough work.

## Part 4. One kernel instead of six

`length(x, y, z)` computes `sqrt(x*x + y*y + z*z)`, the length of a vector in 3D. In CuPy, as in NumPy, every operation is a separate call: three
multiplications, two additions and a square root, six kernels. Each one reads its inputs from the GPU's memory and
writes its result back. `cp.fuse` turns the whole function into one kernel, which reads `x`, `y` and `z` once and
writes the result once.

**Predict**: how many times faster is the fused version?

In [ ]:
!sbatch -p gpu --gres=gpu:1 -c 4 -t 5 -o fuse.out --wait --wrap "python gpu_lab.py fuse"
!cat fuse.out

On julia: 3.81 ms against 1.07 ms, 3.5 times faster, and NumPy 176 ms. Both versions move data at the same rate,
about 1.5 GB per ms: the time follows the bytes read and written, 5.6 GB against 1.6 GB. For element-wise
work, the GPU's memory is the limit, not its cores.

## Part 5. float32 or float64?

First a question for your own session, no GPU needed. `np.cumsum` adds the numbers one after the other, keeping the
running total in float32, as a loop would.

**Predict**: 100 million random numbers between 0 and 1 add up to about 50 million. What will the running float32
sum say?

In [ ]:
import numpy as np

x = np.random.default_rng(1).random(100_000_000, dtype=np.float32)
print(f"running float32 sum: {np.cumsum(x)[-1]:16,.1f}")
print(f"np.sum in float32:   {x.sum():16,.1f}")
print(f"sum in float64:      {x.sum(dtype=np.float64):16,.1f}")

The running sum stops at 16,777,216, which is 2 to the power 24. A float32 has 24 bits for the digits of a number,
so above 16,777,216 the next float32 is 16,777,218. Adding a number below 1 rounds back to the same total, and the sum
stops growing. `np.sum` adds in pairs, then pairs of pairs: its partial sums stay small, and it is right to 8 digits.

Now the speed. The GPUs of our cluster are built for float32: they have few float64 units.

**Predict**: how many times faster is a float32 matrix product than a float64 one on the GPU?

In [ ]:
!sbatch -p gpu --gres=gpu:1 -c 4 -t 5 -o precision.out --wait --wrap "python gpu_lab.py precision"
!cat precision.out

On julia: 65 against 1.5 TFLOP/s, about 40 times. The float32 product differs from the float64 one by about one part
in a million. Whether that is good enough depends on your problem. Check it: run a small case in float64 on the CPU
and compare.

## Part 6. An example from a project (Exercise 3)

The brief suggests a recommendation project: the similarity between every pair of items in a table of ratings. Here,
50,000 users and 5,000 items, 1 GB in float32. The function `similarity` is one function for NumPy and CuPy:
`cp.get_array_module(x)` returns the module that `x` belongs to.

Most of the work is one matrix product, `xn.T @ xn`, of 5,000 x 50,000 by 50,000 x 5,000. The job asks for 8 GB of
memory.

**Predict**: with the copies in and out counted, how many times faster than 4 CPUs?

In [ ]:
!sbatch -p gpu --gres=gpu:1 -c 4 -t 5 --mem=8G -o similarity.out --wait --wrap "python gpu_lab.py similarity"
!cat similarity.out

On julia: 2.99 s on 4 CPUs, 0.98 s on 16 CPUs (another job), and 0.11 s on the GPU with the copies (65 ms in, 36 ms
of work, 8 ms out). The script warms up the copy first: the first big copy of a program costs about 0.2 s more. A lot of work for every byte copied: this is the kind of problem a GPU is built for.

**Exercise 3** (homework):

1. Change `ratings()` to 20,000 items (4 GB). **Predict** the CPU time and the GPU time first: the work grows with
   the square of the items. Then ask for `--mem=24G` and run it.
2. In `precision()`, time `x.sum()` in float32 and in float64 with `gpu_ms`. Is the float64 sum 40 times slower too?
   Why not? (Hint: Part 4.)
3. In `size()`, add a column with the time to copy `a` and `b` to the GPU. With the copies, does the GPU ever win
   for `a + b`?